# NER Ensemble on TEST SET — Final Submission

Loads the two prediction files produced by `bert_NER_inference_TEST_two_models.ipynb`,
applies the best ensemble strategy (**F — Hybrid all labels**: PubMedBERT base + non-overlapping BioBERT spans),
and saves the final submission-ready JSON.

**No GPU needed. No evaluation (no ground truth).**

| Input file | Model |
|------------|-------|
| `pred_NB1b_gold_silver_bronze_TEST.json` | BioBERT (NB1b) |
| `pred_NB1b_pubmed_gold_silver_bronze_TEST.json` | PubMedBERT (NB1b_pubmed) |

Output: `SMTE_T611_R1_NERensemble.json` — ready for submission folder.

## 0. Imports & paths

In [1]:
import json
import copy
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / 'data').exists() and (p / 'src').exists():
            return p
    raise FileNotFoundError('Cannot find repo root')

PROJECT_ROOT = find_repo_root(Path.cwd())
PRED_DIR     = PROJECT_ROOT / 'src' / 'ner' / 'predictions' / 'test_set'

PRED_A_PATH  = PRED_DIR / 'pred_NB1b_gold_silver_bronze_TEST.json'         # BioBERT
PRED_B_PATH  = PRED_DIR / 'pred_NB1b_pubmed_gold_silver_bronze_TEST.json'  # PubMedBERT

# ── Output: final submission JSON ─────────────────────────────────────────────
# ⚠ Rename teamID / runID to match your registered identifiers before submitting
TEAM_ID      = "SMTE"      # ← your CLEF 2026 team ID
TASK_ID      = "T611"
RUN_ID       = "R1"
SYSTEM_DESC  = "NERensemble"
FOLDER_NAME  = f"{TEAM_ID}_{TASK_ID}_{RUN_ID}_{SYSTEM_DESC}"
OUT_FILENAME = f"{FOLDER_NAME}.json"

OUT_DIR = PRED_DIR / FOLDER_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / OUT_FILENAME

print('Project root   :', PROJECT_ROOT)
print('Pred A (BioBERT)  :', PRED_A_PATH.exists(), '—', PRED_A_PATH.name)
print('Pred B (PubMedBERT):', PRED_B_PATH.exists(), '—', PRED_B_PATH.name)
print('Output folder  :', OUT_DIR)
print('Output file    :', OUT_FILENAME)

Project root   : C:\Users\super\Documents\UniPd\ATA\GutBrainIE
Pred A (BioBERT)  : True — pred_NB1b_gold_silver_bronze_TEST.json
Pred B (PubMedBERT): True — pred_NB1b_pubmed_gold_silver_bronze_TEST.json
Output folder  : C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\ner\predictions\test_set\SMTE_T611_R1_NERensemble
Output file    : SMTE_T611_R1_NERensemble.json


## 1. Load predictions

In [2]:
with PRED_A_PATH.open(encoding='utf-8') as f:
    pred_biobert = json.load(f)
with PRED_B_PATH.open(encoding='utf-8') as f:
    pred_pubmed = json.load(f)

print(f'BioBERT   — PMIDs: {len(pred_biobert):4d}  entities: {sum(len(v["entities"]) for v in pred_biobert.values())}')
print(f'PubMedBERT— PMIDs: {len(pred_pubmed):4d}  entities: {sum(len(v["entities"]) for v in pred_pubmed.values())}')

# Sanity check: same PMIDs in both files
assert set(pred_biobert.keys()) == set(pred_pubmed.keys()), "PMID mismatch between prediction files!"
print(f'\n✓ PMIDs match: {len(pred_biobert)} articles')

BioBERT   — PMIDs:   80  entities: 2554
PubMedBERT— PMIDs:   80  entities: 2422

✓ PMIDs match: 80 articles


## 2. Official dedup + overlap removal (from evaluate.py)

Applied **after** ensemble to guarantee submission compliance:
- dedup key = `(start_idx, end_idx, location)` — no label
- overlap = `start_idx < current_end` (strict)

In [3]:
def remove_duplicated_entities(predictions: dict) -> None:
    """In-place. Key = (start_idx, end_idx, location) — no label."""
    removed = 0
    for pmid in list(predictions):
        seen, deduped = set(), []
        for e in predictions[pmid]['entities']:
            k = (e['start_idx'], e['end_idx'], e['location'])
            if k not in seen: seen.add(k); deduped.append(e)
            else: removed += 1
        predictions[pmid]['entities'] = deduped
    if removed > 0:
        print(f'=== Removed {removed} duplicated entities ===')


def remove_overlapping_entities(predictions: dict) -> None:
    """In-place. Overlap condition: start_idx < current_end (strict)."""
    removed = 0
    for pmid in list(predictions):
        orig = len(predictions[pmid]['entities'])
        groups = {'title': [], 'abstract': []}
        for e in predictions[pmid]['entities']:
            groups[e['location']].append(e)
        keepers = set()
        for loc in groups:
            group = sorted(groups[loc], key=lambda e: e['start_idx'])
            clusters, cluster, cur_end = [], [], None
            for e in group:
                if not cluster:
                    cluster = [e]; cur_end = e['end_idx']
                elif e['start_idx'] < cur_end:
                    cluster.append(e)
                    if e['end_idx'] > cur_end: cur_end = e['end_idx']
                else:
                    clusters.append(cluster); cluster = [e]; cur_end = e['end_idx']
            if cluster: clusters.append(cluster)
            for cl in clusters:
                longest = cl[0]
                for e in cl[1:]:
                    if e['end_idx'] - e['start_idx'] > longest['end_idx'] - longest['start_idx']:
                        longest = e
                keepers.add((longest['start_idx'], longest['end_idx'], longest['location']))
        deduped = []
        for e in predictions[pmid]['entities']:
            k = (e['start_idx'], e['end_idx'], e['location'])
            if k in keepers: deduped.append(e); keepers.discard(k)
        predictions[pmid]['entities'] = deduped
        removed += orig - len(deduped)
    if removed > 0:
        print(f'=== Removed {removed} overlapping entities ===')


print('Official dedup/overlap functions ready.')

Official dedup/overlap functions ready.


## 3. Ensemble — Strategy F: Hybrid all labels

PubMedBERT as **base** → add non-overlapping BioBERT spans for **all labels**.

This is the best strategy on dev (Macro-F1 = 0.802, Micro-F1 = 0.8324).

In [4]:
def build_hybrid_all(base: dict, extra: dict) -> dict:
    """
    Strategy F — Hybrid all labels.
    Base model (PubMedBERT) kept as-is.
    Add spans from extra (BioBERT) that don't overlap with any base span.
    """
    result = {}
    for pmid in base:
        covered = set(
            (e['start_idx'], e['end_idx'], e['location'])
            for e in base[pmid]['entities']
        )
        new_spans = [
            e for e in extra[pmid]['entities']
            if (e['start_idx'], e['end_idx'], e['location']) not in covered
        ]
        result[pmid] = {'entities': base[pmid]['entities'] + new_spans}
    return result


# Apply: PubMedBERT base + BioBERT extras
ensemble_preds = build_hybrid_all(pred_pubmed, pred_biobert)

total_before = sum(len(v['entities']) for v in ensemble_preds.values())
print(f'Entities before dedup/overlap removal: {total_before}')

remove_duplicated_entities(ensemble_preds)
remove_overlapping_entities(ensemble_preds)

total_after = sum(len(v['entities']) for v in ensemble_preds.values())
print(f'Entities after  dedup/overlap removal: {total_after}')

Entities before dedup/overlap removal: 2703
=== Removed 64 overlapping entities ===
Entities after  dedup/overlap removal: 2639


## 4. Submission format validation

Checks that:
- every entry has only `entities` (no `uri`, no extra fields)
- every entity has exactly: `start_idx`, `end_idx`, `location`, `text_span`, `label`
- `label` is in the 13 legal values
- `location` is `title` or `abstract`
- no `score` field leaked through

In [5]:
LEGAL_ENTITY_LABELS = {
    'anatomical location', 'animal', 'bacteria', 'biomedical technique',
    'chemical', 'DDF', 'dietary supplement', 'drug', 'food', 'gene',
    'human', 'microbiome', 'statistical technique'
}
REQUIRED_FIELDS = {'start_idx', 'end_idx', 'location', 'text_span', 'label'}
FORBIDDEN_FIELDS = {'uri', 'score'}  # uri → NERD only; score → internal

errors = []
for pmid, obj in ensemble_preds.items():
    if set(obj.keys()) != {'entities'}:
        errors.append(f"{pmid}: unexpected top-level keys {set(obj.keys())}")
    for i, e in enumerate(obj['entities']):
        missing = REQUIRED_FIELDS - set(e.keys())
        if missing:
            errors.append(f"{pmid}[{i}]: missing fields {missing}")
        forbidden = FORBIDDEN_FIELDS & set(e.keys())
        if forbidden:
            errors.append(f"{pmid}[{i}]: forbidden fields {forbidden}")
        if e.get('label') not in LEGAL_ENTITY_LABELS:
            errors.append(f"{pmid}[{i}]: illegal label '{e.get('label')}'")
        if e.get('location') not in {'title', 'abstract'}:
            errors.append(f"{pmid}[{i}]: illegal location '{e.get('location')}'")

if errors:
    print(f'\n⚠️  {len(errors)} validation error(s):')
    for err in errors[:20]:
        print(' ', err)
else:
    print(f'✓ Validation passed — {len(ensemble_preds)} articles, '
          f'{sum(len(v["entities"]) for v in ensemble_preds.values())} entities.')

✓ Validation passed — 80 articles, 2639 entities.


## 5. Save submission JSON

In [6]:
with OUT_PATH.open('w', encoding='utf-8') as f:
    json.dump(ensemble_preds, f, ensure_ascii=False, indent=2)

print(f'✓ Saved: {OUT_PATH}')
print(f'  Folder : {OUT_DIR.name}')
print(f'  File   : {OUT_FILENAME}')

✓ Saved: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\ner\predictions\test_set\SMTE_T611_R1_NERensemble\SMTE_T611_R1_NERensemble.json
  Folder : SMTE_T611_R1_NERensemble
  File   : SMTE_T611_R1_NERensemble.json


## 6. Write .meta file

Required alongside the JSON in the submission folder.

In [7]:
META_CONTENT = f"""Team ID:         {TEAM_ID}
Task ID:         {TASK_ID}
Run ID:          {RUN_ID}

Type of training:
  Fine-tuning of pre-trained transformer models (BioBERT + PubMedBERT) on GutBrainIE 2026
  training data with token classification (BIO tagging, 27 labels).

Pre-processing methods:
  - Separate inference on title and abstract segments to preserve character offsets.
  - BIO repair for malformed I-tags at sequence start.
  - Two-pass thresholding: high-precision pass + recall-boost pass for
    (chemical, bacteria, food, dietary supplement) labels.
  - Label-aware post-processing: gene/chemical remap, food/dietary supplement remap,
    FP filtering based on span text.
  - Span score = median token probability (more robust than mean).
  - Deduplication and soft overlap pruning.
  - Final dedup and overlap removal following the official evaluate.py logic.

Training data used:
  Gold + Silver + Bronze annotations from GutBrainIE 2026 training set.
  Priority merge: when same PMID appears in multiple quality levels,
  higher-quality annotation is kept.

Relevant details of the run:
  This run is a post-hoc hybrid ensemble of two NB1b models:
    - NB1b-BioBERT  (dmis-lab/biobert-v1.1)
    - NB1b-PubMedBERT (microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract)
  Both fine-tuned on Gold+Silver+Bronze with class-weighted cross-entropy,
  lr=3e-5, 5 epochs, batch size 8 (grad accum 2 steps).
  Ensemble strategy F — Hybrid all labels:
    PubMedBERT predictions kept as base;
    non-overlapping BioBERT spans added for all labels.
  Dev set results: Macro-F1=0.802, Micro-F1=0.8324.

GitHub repository:
  https://github.com/sofiamaule/GutBrainIE.git
"""

meta_path = OUT_DIR / f"{FOLDER_NAME}.meta"
meta_path.write_text(META_CONTENT, encoding='utf-8')
print(f'✓ Saved: {meta_path}')
print()
print('── Submission folder contents ──')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name}')

✓ Saved: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\ner\predictions\test_set\SMTE_T611_R1_NERensemble\SMTE_T611_R1_NERensemble.meta

── Submission folder contents ──
  SMTE_T611_R1_NERensemble.json
  SMTE_T611_R1_NERensemble.meta


## 7. Quick stats

In [8]:
from collections import Counter

label_counts = Counter()
for obj in ensemble_preds.values():
    for e in obj['entities']:
        label_counts[e['label']] += 1

print(f'Total articles  : {len(ensemble_preds)}')
print(f'Total entities  : {sum(label_counts.values())}')
print()
print('Per-label counts:')
for lab, cnt in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f'  {lab:<30s}  {cnt:5d}')

Total articles  : 80
Total entities  : 2639

Per-label counts:
  DDF                               789
  chemical                          354
  bacteria                          264
  human                             235
  microbiome                        216
  animal                            156
  anatomical location               142
  drug                              120
  biomedical technique              117
  gene                               89
  dietary supplement                 88
  statistical technique              35
  food                               34
